In [0]:
# Objetivo:
# Importar as bibliotecas necessárias
# para leitura, manipulação e
# persistência dos dados.

# Justificativa:
# As bibliotecas importadas serão
# utilizadas ao longo de todo o
# processo de construção da camada
# Silver.

# Ação:
# Importa as bibliotecas utilizadas
# neste notebook.

import json
from pathlib import Path

import pandas as pd

In [0]:
# Objetivo:
# Definir as configurações utilizadas
# durante a execução do notebook.

# Justificativa:
# Os caminhos são centralizados no
# config.json e na silver_metadata,
# garantindo consistência entre as
# camadas Bronze e Silver.

# Ação:
# Carrega a configuração oficial do
# projeto e filtra os metadados da
# entidade processada neste notebook.

CONFIG_FILE_PATH = "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/config/config.json"

config = json.loads(
    Path(CONFIG_FILE_PATH).read_text(
        encoding="utf-8"
    )
)

BASE_PATH = Path(
    config["environment"]["base_path"]
)

CONFIG_PATH = Path(
    config["paths"]["config_path"]
)

LOG_PATH = Path(
    config["paths"]["log_path"]
)

EXECUTION_DATE = (
    config["project"]["execution_date"]
)

SILVER_METADATA_PATH = (
    CONFIG_PATH
    / "silver_metadata"
)

df_silver_metadata = pd.read_parquet(
    SILVER_METADATA_PATH
)

metadata_dataset = (
    df_silver_metadata[
        df_silver_metadata["dataset"]
        == "metas_ufs"
    ]
    .sort_values("ano")
    .reset_index(drop=True)
)

if set(metadata_dataset["ano"]) != {2023, 2024, 2025}:
    raise ValueError(
        "Metadados Silver incompletos "
        "para o dataset metas_ufs."
    )

CAMINHO_BRONZE = Path(
    metadata_dataset.iloc[0][
        "bronze_path"
    ]
).parent

CAMINHO_SILVER = Path(
    metadata_dataset.iloc[0][
        "silver_path"
    ]
).parent

print(
    "CAMINHO_BRONZE:",
    CAMINHO_BRONZE
)

print(
    "CAMINHO_SILVER:",
    CAMINHO_SILVER
)

display(metadata_dataset)

# 1. Auditoria da Fonte de Dados - Base Metas por UF

> **Nota**
>
> Durante o desenvolvimento deste projeto foi utilizado o dicionário oficial
> dos Microdados da Avaliação da Alfabetização disponibilizado pelo INEP como
> referência para interpretação das variáveis, domínios e regras de negócio
> presentes nas bases de dados.

## 1.1 Leitura das Bases

**Contexto**

Os dados da entidade **Metas por UF** foram disponibilizados pelo INEP em
partições Parquet geradas pela camada Bronze, separados por ano e organizados em múltiplas
abas.

Nesta etapa, os partições são carregadas para o ambiente de análise,
preservando sua estrutura original para que seja possível realizar a
auditoria da qualidade dos dados antes da aplicação das transformações da
camada Silver.

**Objetivo**

Realizar a leitura das bases da entidade Metas por UF referentes aos anos
de 2023, 2024 e 2025.

**Resultado esperado**

Disponibilizar os dados dos três anos em DataFrames independentes,
mantendo a estrutura original dos arquivos da camada Bronze para as etapas
de auditoria e transformação da camada Silver.

### Integração com o Silver Orquestrador

Este notebook não utiliza caminhos locais ou nomes de arquivos fixos.

Os caminhos da entidade `metas_ufs` são obtidos da tabela:

```text
config/silver_metadata
```

A leitura é feita diretamente das partições Parquet da Bronze:

```text
bronze/metas_ufs/ano=2023
bronze/metas_ufs/ano=2024
bronze/metas_ufs/ano=2025
```

A lógica de auditoria, limpeza e transformação construída originalmente permanece preservada.

In [0]:
# Objetivo:
# Carregar as partições anuais da
# entidade metas_ufs na camada Bronze.

# Justificativa:
# A Bronze foi persistida em Parquet,
# organizada por entidade e ano.
# A Silver deve consumir diretamente
# essas partições governadas pelo
# silver_metadata.

# Ação:
# Lê as partições Bronze referentes
# aos anos de 2023, 2024 e 2025.

def caminho_bronze_ano(ano):
    registro = metadata_dataset[
        metadata_dataset["ano"] == ano
    ].iloc[0]

    return Path(
        registro["bronze_path"]
    )


df_metas_ufs_2023 = pd.read_parquet(
    caminho_bronze_ano(2023)
)

df_metas_ufs_2024 = pd.read_parquet(
    caminho_bronze_ano(2024)
)

df_metas_ufs_2025 = pd.read_parquet(
    caminho_bronze_ano(2025)
)

In [0]:

# Objetivo:
# Remover registros que não fazem
# parte da base de dados.

# Justificativa:
# Os arquivos do INEP possuem, ao
# final da planilha, linhas contendo
# observações textuais que não
# representam registros da tabela.
# Sua remoção evita interferências na
# inferência dos tipos de dados e nas
# etapas de auditoria da camada
# Silver.

# Ação:
# Remove as linhas cujo campo ANO não
# representa um valor numérico.

def remover_observacoes(df):

    return (
        df[
            pd.to_numeric(
                df["ANO"],
                errors="coerce"
            ).notna()
        ]
        .copy()
    )


df_metas_ufs_2023 = remover_observacoes(
    df_metas_ufs_2023
)

df_metas_ufs_2024 = remover_observacoes(
    df_metas_ufs_2024
)

df_metas_ufs_2025 = remover_observacoes(
    df_metas_ufs_2025
)

m
## 1.2 Inspeção Inicial da Estrutura

**Contexto**

Antes de realizar qualquer transformação, é importante conhecer a estrutura
das bases disponibilizadas pelo INEP e verificar se houve alterações no
esquema entre os diferentes anos da avaliação.

Essa inspeção permite identificar novas colunas, alterações de nomenclatura
e outras mudanças estruturais que deverão ser consideradas durante a
padronização da camada Silver.

**Objetivo**

Realizar uma inspeção inicial da estrutura das bases de Metas por UF.

**Resultado esperado**

Identificar possíveis diferenças estruturais entre as bases de 2023, 2024
e 2025, subsidiando as etapas de padronização da camada Silver.

In [0]:
# Objetivo:
# Realizar uma inspeção inicial da
# estrutura das bases de Metas por
# UF.

# Justificativa:
# A inspeção inicial permite verificar
# o esquema das bases e identificar
# possíveis alterações nas colunas
# disponibilizadas pelo INEP ao longo
# dos anos da avaliação.

# Ação:
# Exibe a estrutura das bases de
# Metas por UF para comparação entre
# os anos de 2023, 2024 e 2025.

print(df_metas_ufs_2023.columns.tolist())

print(df_metas_ufs_2024.columns.tolist())

print(df_metas_ufs_2025.columns.tolist())

## 1.3 Comparação dos Tipos de Dados

**Contexto**

Após a inspeção inicial da estrutura das bases, é necessário comparar os
tipos de dados dos atributos entre os diferentes anos da avaliação.

Essa análise permite identificar alterações de esquema, inclusão ou remoção
de colunas e possíveis incompatibilidades que deverão ser tratadas durante
a construção da camada Silver.

**Objetivo**

Comparar os tipos de dados das bases de Metas por UF dos anos de 2023,
2024 e 2025.

**Resultado esperado**

Obter um relatório consolidado contendo os tipos de dados de cada coluna e
a classificação de compatibilidade entre as estruturas das três bases.

In [0]:
# Objetivo:
# Comparar os tipos de dados das
# bases de Metas por UF dos anos de
# 2023, 2024 e 2025.

# Justificativa:
# Colunas com o mesmo nome podem
# apresentar tipos diferentes entre
# os anos ou novas variáveis podem
# ter sido incorporadas pelo INEP.

# Ação:
# Consolida os tipos de dados das
# três bases em um único relatório
# e classifica a compatibilidade
# entre os esquemas.

relatorio_dtypes = pd.DataFrame({
    "2023": df_metas_ufs_2023.dtypes.astype(str),
    "2024": df_metas_ufs_2024.dtypes.astype(str),
    "2025": df_metas_ufs_2025.dtypes.astype(str)
})

relatorio_dtypes = relatorio_dtypes.replace("nan", pd.NA)


def classificar_status(linha):

    if (
        pd.notna(linha["2023"])
        and pd.isna(linha["2024"])
        and pd.isna(linha["2025"])
    ):
        return "Exclusiva de 2023"

    if (
        pd.isna(linha["2023"])
        and pd.notna(linha["2024"])
        and pd.notna(linha["2025"])
    ):
        return "Nova a partir de 2024"

    if (
        pd.isna(linha["2023"])
        and pd.isna(linha["2024"])
        and pd.notna(linha["2025"])
    ):
        return "Nova em 2025"

    tipos = linha.dropna()

    if len(tipos.unique()) == 1:
        return "Compatível"

    return "Divergente"


relatorio_dtypes["status"] = relatorio_dtypes.apply(
    classificar_status,
    axis=1
)

display(relatorio_dtypes)

In [0]:
# Objetivo:
# Identificar os valores que estão
# provocando divergências nos tipos
# de dados entre as bases.

# Justificativa:
# Valores textuais utilizados para
# representar ausência de informação
# podem alterar a inferência dos
# tipos de dados realizada pelo
# Pandas.

# Ação:
# Exibe os valores não numéricos das
# colunas classificadas como
# divergentes.

colunas = [
    "ANO",
    "META_FINAL_2024",
    "META_FINAL_2025",
    "META_FINAL_2026",
    "META_FINAL_2027",
    "META_FINAL_2028",
    "META_FINAL_2029",
    "PC_AVALIADOS_LP"
]

for coluna in colunas:

    print(f"\n{coluna}")

    for ano, df in [
        (2023, df_metas_ufs_2023),
        (2024, df_metas_ufs_2024),
        (2025, df_metas_ufs_2025)
    ]:

        valores = (
            df.loc[
                pd.to_numeric(
                    df[coluna],
                    errors="coerce"
                ).isna(),
                coluna
            ]
            .drop_duplicates()
            .tolist()
        )

        print(f"{ano}: {valores}")

## 1.4 Análise dos Valores Ausentes

**Contexto**

Após a verificação da estrutura e dos tipos de dados, torna-se necessário
avaliar a presença de valores ausentes nas bases dos diferentes anos.

Durante a auditoria foi identificado que as bases utilizam o marcador
textual `"-"` para representar ausência de informação, além do valor
`">80"` nas colunas de metas, que representa uma regra de negócio
estabelecida pelo INEP e não um valor ausente.

**Objetivo**

Identificar e quantificar a ocorrência de valores ausentes nas bases da
entidade Metas por UF, preservando a interpretação correta dos indicadores
disponibilizados pelo INEP.

**Resultado esperado**

Obter um diagnóstico da completude dos dados, distinguindo valores
ausentes de regras de negócio representadas por marcadores textuais.

In [0]:
# Objetivo:
# Analisar a ocorrência de valores
# ausentes nas bases da entidade
# Metas por UF.

# Justificativa:
# As bases utilizam o marcador "-"
# para representar ausência de
# informação. Já o valor ">80"
# representa uma regra de negócio do
# INEP e deve ser preservado durante
# a auditoria.

# Ação:
# Substitui apenas o marcador "-"
# por valores ausentes e calcula a
# quantidade de valores ausentes por
# coluna em cada base.

for ano, df in [
    (2023, df_metas_ufs_2023),
    (2024, df_metas_ufs_2024),
    (2025, df_metas_ufs_2025)
]:

    print(f"\nValores ausentes - {ano}")

    df_analise = (
        df.replace("- ", pd.NA)
          .replace("-", pd.NA)
    )

    valores_ausentes = (
        df_analise
        .isna()
        .sum()
        .loc[lambda s: s > 0]
        .sort_values(ascending=False)
        .to_frame("Valores Ausentes")
    )

    if valores_ausentes.empty:
        print("Nenhum valor ausente encontrado.")
    else:
        display(valores_ausentes)

In [0]:
# Objetivo:
# Identificar os registros que
# apresentam valores ausentes nas
# colunas de identificação da UF.

# Justificativa:
# Valores ausentes em atributos de
# identificação podem representar
# registros agregados divulgados pelo
# INEP, como o total Brasil, e não
# necessariamente inconsistências da
# base.
#
# Quando o filtro não retorna linhas,
# o display() do Databricks pode tentar
# converter um DataFrame Pandas vazio
# para Spark e gerar o erro:
# "Can not infer schema from an empty
# dataset".
#
# Para evitar esse comportamento, a
# exibição será realizada somente
# quando existirem registros.

# Ação:
# Filtra os registros com código ou
# sigla da UF ausentes e exibe apenas
# os resultados não vazios.

for ano, df in [
    (2023, df_metas_ufs_2023),
    (2024, df_metas_ufs_2024),
    (2025, df_metas_ufs_2025)
]:

    print(f"\nBase {ano}")

    df_identificacao_ausente = (
        df.loc[
            df["CD_UF"].isna()
            | df["SIGLA_UF"].isna()
        ]
        .copy()
        .reset_index(drop=True)
    )

    if df_identificacao_ausente.empty:
        print(
            "Nenhum registro com CD_UF "
            "ou SIGLA_UF ausente."
        )
    else:
        display(
            df_identificacao_ausente
        )

In [0]:
# Objetivo:
# Verificar a origem dos valores
# ausentes identificados nas metas
# e nos indicadores de alfabetização.

# Justificativa:
# A análise permite confirmar se os
# valores ausentes pertencem ao mesmo
# conjunto de UFs, indicando uma
# característica da divulgação dos
# dados pelo INEP.
#
# A exibição é condicionada à
# existência de registros para evitar
# erro de inferência de schema em
# DataFrames Pandas vazios.

# Ação:
# Substitui os marcadores de ausência,
# filtra os registros sem percentual
# de alfabetização e exibe somente
# quando houver resultados.

df_indicadores_ausentes_2023 = (
    df_metas_ufs_2023
    .replace("- ", pd.NA)
    .replace("-", pd.NA)
    .loc[
        lambda df: (
            df[
                "PC_ALUNO_ALFABETIZADO"
            ].isna()
        )
    ]
    .copy()
    .reset_index(drop=True)
)

if df_indicadores_ausentes_2023.empty:
    print(
        "2023: nenhum registro com "
        "PC_ALUNO_ALFABETIZADO ausente."
    )
else:
    display(
        df_indicadores_ausentes_2023
    )

### 1.4.1 Análise dos Resultados

A análise identificou valores ausentes nas bases dos anos de 2023, 2024 e
2025.

A investigação demonstrou que essas ausências não representam
inconsistências na qualidade dos dados, mas refletem as regras de
divulgação estabelecidas pelo INEP para a publicação dos resultados da
Avaliação da Alfabetização.

Também foi identificado que as bases utilizam dois marcadores textuais com
significados distintos. O marcador `"-"` representa ausência de informação
e será convertido para valores ausentes reconhecidos pelo Pandas (`pd.NA`)
durante a construção da camada Silver. Já o marcador `">80"` representa
uma regra de negócio definida pelo INEP para unidades da federação que já
atingiram o percentual de alfabetização superior a 80%, devendo ser
preservado como valor textual.

A auditoria confirmou ainda que o registro nacional referente ao Brasil é
apresentado sem código e sigla de unidade da federação, por representar um
resultado agregado nacional e não uma unidade federativa específica.

Dessa forma, as características identificadas serão preservadas durante a
construção da camada Silver, realizando apenas a padronização da
representação dos valores ausentes, sem alterar o significado das
informações disponibilizadas pelo INEP.

## 1.5 Seleção das Colunas da Camada Silver

**Contexto**

A auditoria das bases evidenciou alterações estruturais entre as edições da
Avaliação da Alfabetização, incluindo a evolução dos indicadores de
percentual de alunos alfabetizados e a descontinuidade de alguns atributos
presentes apenas na base de 2023.

Nesta etapa é realizada a padronização da estrutura das bases,
preservando os atributos disponibilizados pelo INEP a partir de 2024 e
garantindo compatibilidade entre todas as partições da camada Silver.

**Objetivo**

Definir o conjunto de atributos que comporá a Base Metas por UF da camada
Silver, padronizando a estrutura das bases de 2023, 2024 e 2025.

**Resultado esperado**

Obter três bases com a mesma estrutura de colunas, preservando a evolução
histórica dos indicadores e garantindo compatibilidade estrutural para as
etapas posteriores do pipeline analítico.

In [0]:
# Objetivo:
# Selecionar as colunas que farão
# parte da Base Metas por UF da
# camada Silver.

# Justificativa:
# As bases sofreram evolução de
# estrutura entre 2023 e 2025,
# incorporando novos indicadores e
# alterando a nomenclatura de alguns
# atributos. A padronização garante
# um esquema único para todas as
# partições da camada Silver.

# Ação:
# Padroniza os nomes das colunas,
# cria os atributos ausentes nas
# bases anteriores, converte o
# marcador "-" para valores
# ausentes, padroniza o marcador
# "> 80" e seleciona o conjunto
# final de atributos da camada
# Silver.

# Padronização da nomenclatura
df_metas_ufs_2023 = (
    df_metas_ufs_2023
    .rename(
        columns={
            "PC_ALUNO_ALFABETIZADO": "PC_ALUNO_ALFABETIZADO_2023"
        }
    )
)

# Estrutura final da camada Silver
colunas_silver = [
    "ANO",
    "CD_UF",
    "SIGLA_UF",
    "NOME_UF",
    "REDE",
    "PC_ALUNO_ALFABETIZADO_2023",
    "PC_ALUNO_ALFABETIZADO_2024",
    "PC_ALUNO_ALFABETIZADO_2025",
    "META_FINAL_2024",
    "META_FINAL_2025",
    "META_FINAL_2026",
    "META_FINAL_2027",
    "META_FINAL_2028",
    "META_FINAL_2029",
    "META_FINAL_2030",
    "PC_AVALIADOS_LP"
]

# Padronização das bases
for df in [
    df_metas_ufs_2023,
    df_metas_ufs_2024,
    df_metas_ufs_2025
]:

    df.replace(
        {
            "-": pd.NA,
            "- ": pd.NA,
            "> 80": ">80"
        },
        inplace=True
    )

    for coluna in colunas_silver:

        if coluna not in df.columns:
            df[coluna] = pd.NA

# Seleção das colunas
df_metas_ufs_2023 = (
    df_metas_ufs_2023[colunas_silver]
    .copy()
)

df_metas_ufs_2024 = (
    df_metas_ufs_2024[colunas_silver]
    .copy()
)

df_metas_ufs_2025 = (
    df_metas_ufs_2025[colunas_silver]
    .copy()
)

## 1.6 Validação da Estrutura da Camada Silver

**Contexto**

Após a padronização da estrutura das bases, torna-se necessário verificar
se as três partições anuais compartilham exatamente o mesmo esquema.

Essa validação garante que todas as bases estejam preparadas para a
persistência na camada Silver e para sua utilização nas etapas posteriores
do pipeline analítico.

**Objetivo**

Validar a estrutura das bases da entidade Metas por UF após o processo de
padronização.

**Resultado esperado**

Confirmar que as bases de 2023, 2024 e 2025 possuem exatamente a mesma
quantidade de colunas, assegurando a compatibilidade estrutural entre as
partições da camada Silver.

In [0]:
# Objetivo:
# Validar a estrutura das bases da
# entidade Metas por UF após o
# processo de padronização.

# Justificativa:
# A validação confirma que todas as
# partições da camada Silver possuem
# exatamente o mesmo esquema antes
# da persistência dos dados.

# Ação:
# Compara a quantidade de colunas das
# três bases e classifica o resultado
# da validação.

validacao = pd.DataFrame({
    "Base": ["2023", "2024", "2025"],
    "Quantidade de Colunas": [
        len(df_metas_ufs_2023.columns),
        len(df_metas_ufs_2024.columns),
        len(df_metas_ufs_2025.columns)
    ]
})

quantidade_esperada = len(colunas_silver)

validacao["Status"] = validacao[
    "Quantidade de Colunas"
].apply(
    lambda x: (
        "Compatível"
        if x == quantidade_esperada
        else "Incompatível"
    )
)

display(validacao)

## 1.7 Persistência da Base Metas por UF

**Contexto**

Após a validação da estrutura das bases, os dados encontram-se prontos para
serem persistidos na camada Silver.

Nesta etapa, cada partição anual é armazenada em seu respectivo diretório,
preservando a organização por entidade e por ano definida para a
arquitetura do projeto.

**Objetivo**

Persistir as bases da entidade Metas por UF na camada Silver, mantendo o
particionamento anual.

**Resultado esperado**

Armazenar as bases tratadas da entidade Metas por UF na camada Silver,
preservando a estrutura padronizada e a organização do Data Lake para as
etapas posteriores do pipeline analítico.

In [0]:
# Objetivo:
# Persistir as bases da entidade
# metas_ufs na camada Silver.

# Justificativa:
# A persistência utiliza os caminhos
# e nomes de arquivo registrados na
# silver_metadata, eliminando caminhos
# fixos e mantendo o particionamento
# anual definido no setup.

# Ação:
# Grava as bases tratadas dos anos de
# 2023, 2024 e 2025 em CSV UTF-8.

for ano, df in [
    (2023, df_metas_ufs_2023),
    (2024, df_metas_ufs_2024),
    (2025, df_metas_ufs_2025)
]:
    registro = metadata_dataset[
        metadata_dataset["ano"] == ano
    ].iloc[0]

    destino = Path(
        registro["silver_path"]
    )

    nome_arquivo = registro[
        "silver_file_name"
    ]

    destino.mkdir(
        parents=True,
        exist_ok=True
    )

    df.to_csv(
        destino / nome_arquivo,
        sep=";",
        decimal=",",
        encoding="utf-8",
        index=False
    )

    print(
        f"Silver salva: "
        f"{destino / nome_arquivo}"
    )

# Conclusão

Ao longo deste notebook foi realizada a auditoria estrutural das bases da
entidade **Metas por UF** referentes aos anos de **2023**, **2024** e
**2025**, identificando a evolução do esquema de dados e as regras de
negócio adotadas pelo INEP para divulgação dos indicadores da Avaliação da
Alfabetização.

A auditoria contemplou a verificação da estrutura das bases, a comparação
dos tipos de dados e a análise dos valores ausentes. Durante esse processo,
foi identificado que o marcador textual `"-"` representa ausência de
informação, sendo convertido para valores ausentes reconhecidos pelo Pandas
(`pd.NA`). Também foi identificado o marcador `">80"`, utilizado pelo
INEP para indicar unidades da federação que já alcançaram percentual de
alfabetização superior a 80%, sendo preservado como informação textual por
representar uma regra de negócio da fonte oficial.

A análise também confirmou que o registro referente ao Brasil representa um
resultado agregado nacional, justificando a ausência de código e sigla de
unidade da federação sem caracterizar inconsistência nos dados.

Foi identificada ainda a evolução da estrutura das bases ao longo dos
anos, com a substituição do indicador único de percentual de alunos
alfabetizados por colunas específicas para cada ano de avaliação. Em
contrapartida, os indicadores históricos `SAEB_2019` e `SAEB_2021`,
presentes apenas na base de 2023, deixaram de compor a estrutura oficial
das bases posteriores e, por esse motivo, não foram incorporados ao esquema
padronizado da camada Silver.

Após a padronização estrutural e a validação da compatibilidade entre as
partições anuais, as bases foram persistidas na camada **Silver**,
mantendo a organização por entidade e o particionamento por ano definido
na arquitetura do projeto.

Dessa forma, a Base Metas por UF da camada Silver representa uma versão
padronizada, auditada e consistente dos dados oficiais disponibilizados
pelo INEP, preservando sua evolução histórica e fornecendo uma base
confiável para utilização nas etapas analíticas desenvolvidas na camada
Gold.